In [14]:
# ============================================================
# HIERARCHICAL SUMMARIZATION — LOAD + PREPROCESS SOURCE NOTES
#
# Use the same preprocessing as the other workflows so all
# summarization methods operate on the same source records.
# ============================================================

import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

In [15]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..", "..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

In [16]:
from config.prompts import SYSTEM_PROMPT
from src.llm.llm import generate_summary

In [17]:
PROJECT_ROOT = Path("../..").resolve()

NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

notes = pd.read_csv(NOTES_PATH)

notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()

notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text",
        ],
        keep="first",
    )
    .reset_index(drop=True)
)

print("Raw notes:", len(notes))
print("After cleaning:", len(notes_clean))
print("After deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

Raw notes: 1602
After cleaning: 1595
After deduplication: 1103
Patients: 50


In [18]:
# ============================================================
# SANITY-TEST PATIENT
# ============================================================

SELECTED_PERSON_ID = (
    "c87e610e-ac2a-48cf-ac32-054f3e595498"
)

patient_notes = (
    notes_dedup[
        notes_dedup["person_id"]
        == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Patient:", SELECTED_PERSON_ID)
print("Deduplicated notes:", len(patient_notes))

Patient: c87e610e-ac2a-48cf-ac32-054f3e595498
Deduplicated notes: 10


In [19]:
# ============================================================
# LEVEL 1 — NOTE-LEVEL SUMMARIZATION PROMPT
#
# Each source note is summarized independently before the
# patient-level synthesis step.
# ============================================================

NOTE_SUMMARY_PROMPT = """
You are summarizing one clinical note from a longitudinal patient record.

Create a concise clinical summary of this note.

Include only clinically meaningful information explicitly documented
in the note, such as:
- presenting symptoms or complaints
- diagnoses or clinical assessments
- examination findings
- investigations and results
- medications and treatments
- procedures or interventions
- clinically meaningful progression or response
- referrals, disposition, or follow-up plans

Rules:
- Do not add or infer information.
- Do not introduce new diagnoses, medications, or findings.
- Preserve clinically important values, doses, and findings.
- Preserve temporal information when explicitly stated.
- Remove administrative or repetitive wording unless clinically relevant.
- If the note contains very little clinical information, keep the summary brief.

Return only the note summary.
"""

In [20]:
# ------------------------------------------------------------
# Generate one intermediate summary from one clinical note.
# ------------------------------------------------------------

from src.llm.llm import LLM_MODEL, _get_client


def summarize_single_note(note_text):
    """
    Generate a concise clinical summary for one source note.
    """

    client = _get_client()

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": NOTE_SUMMARY_PROMPT,
            },
            {
                "role": "user",
                "content": note_text,
            },
        ],
    )

    return response.choices[0].message.content

In [21]:
# ============================================================
# LEVEL 1 — GENERATE NOTE-LEVEL SUMMARIES
#
# Each completed note is checkpointed immediately so interrupted
# runs can resume without repeating successful LLM calls.
# ============================================================

HIERARCHICAL_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "hierarchical"
)

HIERARCHICAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTE_SUMMARY_PATH = (
    HIERARCHICAL_DIR
    / "sanity_note_summaries.json"
)

if NOTE_SUMMARY_PATH.exists():

    with NOTE_SUMMARY_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        note_summaries = json.load(file)

else:
    note_summaries = {}


for note_index, row in patient_notes.iterrows():

    note_key = str(note_index)

    if note_key in note_summaries:
        print(
            f"Skipping completed note: {note_index}"
        )
        continue

    print(
        f"Summarizing note "
        f"{note_index + 1}/{len(patient_notes)}"
    )

    summary = summarize_single_note(
        row["clean_note_text"]
    )

    note_summaries[note_key] = {
        "source_note_index": int(note_index),
        "creation_timestamp": str(
            row["creation_timestamp"]
        ),
        "summary": summary,
    }

    with NOTE_SUMMARY_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            note_summaries,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("Saved:", note_index)

    time.sleep(1)

Skipping completed note: 0
Skipping completed note: 1
Skipping completed note: 2
Skipping completed note: 3
Skipping completed note: 4
Skipping completed note: 5
Skipping completed note: 6
Skipping completed note: 7
Skipping completed note: 8
Skipping completed note: 9


In [22]:
# ============================================================
# BUILD CHRONOLOGICAL INTERMEDIATE CONTEXT
#
# The note-level summaries are placed back in their original
# chronological order before final patient-level synthesis.
# ============================================================

ordered_note_summaries = sorted(
    note_summaries.values(),
    key=lambda item: item[
        "source_note_index"
    ],
)

hierarchical_context_parts = []

for item in ordered_note_summaries:

    hierarchical_context_parts.append(
        f"[{item['creation_timestamp']}]\n"
        f"{item['summary']}"
    )

hierarchical_context = "\n\n".join(
    hierarchical_context_parts
)

print("Intermediate summaries:", len(ordered_note_summaries))
print("Context characters:", len(hierarchical_context))
print("Context words:", len(hierarchical_context.split()))

print("\n--- PREVIEW ---\n")
print(hierarchical_context[:3000])

Intermediate summaries: 10
Context characters: 5158
Context words: 785

--- PREVIEW ---

[04/01/2026 10:10]
1-year-old female with ear pain. Observations (HR, RR, temperature) were within normal limits, and she was in no acute distress. No allergies or past medical history documented. Directed to triage for further evaluation of ear pain.

[04/01/2026 10:30]
Afebrile with mild erythema around the right outer ear and moderate tenderness on palpation. No systemic signs of illness noted. Plan for doctor review and otoscopic examination to confirm diagnosis and determine treatment.

[04/01/2026 11:00]
Patient reviewed in the ED on 04/01/26. Baseline observations were normal: heart rate, temperature, and SpO2 all within normal limits. Patient was in no acute distress and remained stable. Plan was to await senior review; no medications or interventions were initiated, and the patient was to remain in the ED for further assessment.

[04/01/2026 11:30]
1-year-old female presented with irritabi

In [23]:
# ============================================================
# BUILD FINAL SYNTHESIS INPUT
#
# Convert the chronological note-level summaries into the same
# dataframe structure expected by generate_summary().
# ============================================================

hierarchical_notes_df = pd.DataFrame(
    [
        {
            "person_id": SELECTED_PERSON_ID,
            "creation_timestamp": item["creation_timestamp"],
            "clean_note_text": item["summary"],
        }
        for item in ordered_note_summaries
    ]
)

print("Intermediate summaries:", len(hierarchical_notes_df))

display(
    hierarchical_notes_df[
        [
            "creation_timestamp",
            "clean_note_text",
        ]
    ].head()
)

Intermediate summaries: 10


,creation_timestamp,clean_note_text
0,04/01/2026 10:10,1-year-old female with ear pain. Observations ...
1,04/01/2026 10:30,Afebrile with mild erythema around the right o...
2,04/01/2026 11:00,Patient reviewed in the ED on 04/01/26. Baseli...
3,04/01/2026 11:30,"1-year-old female presented with irritability,..."
4,04/01/2026 12:00,Acute right ear pain for 24 hours with intermi...


In [24]:
# ============================================================
# LEVEL 2 — FINAL HIERARCHICAL SYNTHESIS
#
# Use the same final SYSTEM_PROMPT used by the other workflows.
# ============================================================


hierarchical_summary = generate_summary(
    SELECTED_PERSON_ID,
    hierarchical_notes_df,
    SYSTEM_PROMPT,
)

print(hierarchical_summary)

A 1-year-old female with no documented past medical history or allergies presented with approximately 24 hours of right ear pain, ear tugging, intermittent crying, irritability, decreased feeding, and difficulty sleeping. She had no fever, vomiting, diarrhoea, respiratory symptoms, rash, or other systemic features. Initial observations were normal, and she was afebrile and in no acute distress. Examination showed mild erythema and moderate tenderness of the right outer ear.

On ED assessment (04/01/26), she was alert but irritable. Otoscopy demonstrated a bulging, erythematous right tympanic membrane with effusion; the left ear was normal. There was no mastoiditis, cellulitis, pre-auricular tenderness, facial asymmetry, neurological deficit, neck stiffness, lymphadenopathy, or other evidence of systemic infection. Vital signs remained stable: HR 110 bpm, RR 24–28/min, temperature 36.8°C, and SpO₂ 98–99% on air. She was diagnosed with acute right otitis media with effusion, considered l

In [25]:
# ============================================================
# SAVE SANITY-TEST FINAL HIERARCHICAL SUMMARY
# ============================================================

SANITY_FINAL_PATH = (
    HIERARCHICAL_DIR
    / "sanity_hierarchical_summary.json"
)

sanity_result = {
    "person_id": SELECTED_PERSON_ID,
    "source_note_count": len(patient_notes),
    "intermediate_summary_count": len(
        ordered_note_summaries
    ),
    "intermediate_context_characters": len(
        hierarchical_context
    ),
    "final_summary": hierarchical_summary,
}

with SANITY_FINAL_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sanity_result,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", SANITY_FINAL_PATH)

Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/processed/hierarchical/sanity_hierarchical_summary.json


In [26]:
# ============================================================
# FULL-COHORT OUTPUT PATHS
# ============================================================

NOTE_LEVEL_PATH = (
    HIERARCHICAL_DIR
    / "hierarchical_note_summaries.json"
)

FINAL_RESULTS_PATH = (
    HIERARCHICAL_DIR
    / "hierarchical_summaries.json"
)


# Load existing checkpoints if present.
if NOTE_LEVEL_PATH.exists():
    with NOTE_LEVEL_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        all_note_summaries = json.load(file)
else:
    all_note_summaries = {}


if FINAL_RESULTS_PATH.exists():
    with FINAL_RESULTS_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        hierarchical_results = json.load(file)
else:
    hierarchical_results = {}


print(
    "Patients with note-level progress:",
    len(all_note_summaries),
)

print(
    "Patients with final summaries:",
    len(hierarchical_results),
)

Patients with note-level progress: 20
Patients with final summaries: 20


In [27]:
# ============================================================
# RUN HIERARCHICAL SUMMARIZATION FOR ONE PATIENT
# ============================================================

def run_hierarchical_patient(
    person_id,
    patient_df,
):
    """
    Hierarchical summarization for one patient:

    1. Summarize each deduplicated source note independently.
    2. Preserve note summaries in chronological order.
    3. Generate one final longitudinal summary using SYSTEM_PROMPT.

    Note-level summaries are checkpointed after every successful call.
    """

    patient_df = (
        patient_df
        .sort_values("creation_timestamp")
        .reset_index(drop=True)
    )

    all_note_summaries.setdefault(
        person_id,
        {}
    )

    # --------------------------------------------------------
    # LEVEL 1 — NOTE SUMMARIZATION
    # --------------------------------------------------------

    for note_index, row in patient_df.iterrows():

        note_key = str(note_index)

        # Do not pay for completed notes again.
        if note_key in all_note_summaries[person_id]:
            continue

        print(
            f"  Note {note_index + 1}/"
            f"{len(patient_df)}"
        )

        summary = summarize_single_note(
            row["clean_note_text"]
        )

        all_note_summaries[
            person_id
        ][note_key] = {
            "source_note_index": int(note_index),
            "creation_timestamp": str(
                row["creation_timestamp"]
            ),
            "summary": summary,
        }

        # Checkpoint after every note.
        with NOTE_LEVEL_PATH.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                all_note_summaries,
                file,
                indent=2,
                ensure_ascii=False,
            )

        time.sleep(1)


    # --------------------------------------------------------
    # BUILD CHRONOLOGICAL LEVEL-2 INPUT
    # --------------------------------------------------------

    ordered = sorted(
        all_note_summaries[person_id].values(),
        key=lambda item: item[
            "source_note_index"
        ],
    )

    hierarchical_notes_df = pd.DataFrame(
        [
            {
                "person_id": person_id,
                "creation_timestamp": item[
                    "creation_timestamp"
                ],
                "clean_note_text": item[
                    "summary"
                ],
            }
            for item in ordered
        ]
    )


    # --------------------------------------------------------
    # LEVEL 2 — FINAL LONGITUDINAL SYNTHESIS
    # --------------------------------------------------------

    final_summary = generate_summary(
        person_id,
        hierarchical_notes_df,
        SYSTEM_PROMPT,
    )

    return {
        "person_id": person_id,
        "source_note_count": len(patient_df),
        "intermediate_summary_count": len(ordered),
        "final_summary": final_summary,
    }

In [28]:
# ============================================================
# GENERATE HIERARCHICAL SUMMARIES FOR THE FULL COHORT
#
# Completed patients and completed note-level summaries are
# skipped automatically when this cell is rerun.
# ============================================================

patient_ids = (
    notes_dedup["person_id"]
    .drop_duplicates()
    .tolist()
)

print("Patients to process:", len(patient_ids))


for patient_number, person_id in enumerate(
    patient_ids,
    start=1,
):

    # Never regenerate a completed final summary.
    if person_id in hierarchical_results:
        print(
            f"\n[{patient_number}/{len(patient_ids)}] "
            f"Skipping completed patient: {person_id}"
        )
        continue

    print(
        f"\n{'=' * 70}\n"
        f"[{patient_number}/{len(patient_ids)}] "
        f"Patient: {person_id}\n"
        f"{'=' * 70}"
    )

    patient_df = notes_dedup[
        notes_dedup["person_id"] == person_id
    ].copy()

    try:

        result = run_hierarchical_patient(
            person_id,
            patient_df,
        )

        hierarchical_results[
            person_id
        ] = result

        # Checkpoint final patient result immediately.
        with FINAL_RESULTS_PATH.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                hierarchical_results,
                file,
                indent=2,
                ensure_ascii=False,
            )

        print(
            f"Saved final summary: {person_id}"
        )

        time.sleep(2)

    except Exception as error:

        print(
            f"\nSTOPPED at patient: {person_id}"
        )

        print(
            type(error).__name__,
            ":",
            error,
        )

        # Stop instead of silently losing a patient.
        raise


print(
    "\nCompleted hierarchical summaries:",
    len(hierarchical_results),
)

Patients to process: 50

[1/50] Skipping completed patient: 028998ee-babc-4096-9b28-001bc2f9a84e

[2/50] Skipping completed patient: 04df53ea-55c1-48d9-84a1-1f15c133b29b

[3/50] Skipping completed patient: 05192757-942f-460d-b4ff-004ec39cc5ee

[4/50] Skipping completed patient: 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf

[5/50] Skipping completed patient: 0f438665-d430-4adb-8acc-c3beed9e4942

[6/50] Skipping completed patient: 136c7916-4f9b-4e5c-bf01-77e9d2c681a2

[7/50] Skipping completed patient: 137b8481-4f1d-4b7f-babd-20f7117023ad

[8/50] Skipping completed patient: 1705dd0f-011a-492c-b006-b27e03f2f4ed

[9/50] Skipping completed patient: 1dbe23dc-0d1e-431b-81eb-497282b46a14

[10/50] Skipping completed patient: 28570119-9cdc-4120-98c0-4edb76cf36a3

[11/50] Skipping completed patient: 29ea304f-821d-474e-81a1-394ca3945e02

[12/50] Skipping completed patient: 31f9612b-5a6b-48ea-887b-895772a83b99

[13/50] Skipping completed patient: 359014a1-10e6-4bd8-9ba7-513d021c971e

[14/50] Skipping compl

In [29]:
print("Final hierarchical summaries:", len(hierarchical_results))

assert len(hierarchical_results) == 50

print("✓ Hierarchical workflow complete")

Final hierarchical summaries: 50
✓ Hierarchical workflow complete


In [30]:
# ============================================================
# FREEZE STUDY COHORT FROM COMPLETED HIERARCHICAL PATIENTS
#
# These exact patients will be used for all four workflows.
# ============================================================

FINAL_RESULTS_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "hierarchical"
    / "hierarchical_summaries.json"
)

with FINAL_RESULTS_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    completed_hierarchical = json.load(file)

STUDY_PATIENT_IDS = list(
    completed_hierarchical.keys()
)

print(
    "Completed hierarchical patients:",
    len(STUDY_PATIENT_IDS)
)

for i, person_id in enumerate(
    STUDY_PATIENT_IDS,
    start=1,
):
    note_count = (
        notes_dedup["person_id"]
        == person_id
    ).sum()

    print(
        f"{i:02d} | {person_id} | "
        f"{note_count} notes"
    )

Completed hierarchical patients: 50
01 | 028998ee-babc-4096-9b28-001bc2f9a84e | 15 notes
02 | 04df53ea-55c1-48d9-84a1-1f15c133b29b | 33 notes
03 | 05192757-942f-460d-b4ff-004ec39cc5ee | 23 notes
04 | 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf | 19 notes
05 | 0f438665-d430-4adb-8acc-c3beed9e4942 | 15 notes
06 | 136c7916-4f9b-4e5c-bf01-77e9d2c681a2 | 37 notes
07 | 137b8481-4f1d-4b7f-babd-20f7117023ad | 36 notes
08 | 1705dd0f-011a-492c-b006-b27e03f2f4ed | 14 notes
09 | 1dbe23dc-0d1e-431b-81eb-497282b46a14 | 17 notes
10 | 28570119-9cdc-4120-98c0-4edb76cf36a3 | 19 notes
11 | 29ea304f-821d-474e-81a1-394ca3945e02 | 17 notes
12 | 31f9612b-5a6b-48ea-887b-895772a83b99 | 22 notes
13 | 359014a1-10e6-4bd8-9ba7-513d021c971e | 34 notes
14 | 37b5ce4d-dcfd-4bb7-bee4-d597eb114703 | 26 notes
15 | 420df33b-0072-4124-b1c6-0589daef3677 | 24 notes
16 | 42149ae1-6a3c-471e-a002-cb7263e8bb8c | 25 notes
17 | 51f15281-8840-4fd0-92de-89188ab8d736 | 32 notes
18 | 58b8aad6-7327-4450-956f-b775be0f4984 | 11 notes
19 | 5cfa1

In [ ]:
# ============================================================
# SAVE FROZEN 20-PATIENT STUDY COHORT
# ============================================================

COHORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "patients"
    / "study_patient_ids.json"
)

COHORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with COHORT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        STUDY_PATIENT_IDS,
        file,
        indent=2,
    )

print("Study patients:", len(STUDY_PATIENT_IDS))
print("Saved:", COHORT_PATH)

Study patients: 19
Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/patients/study_patient_ids.json


In [ ]:
# ============================================================
# FIND NEXT UNFINISHED PATIENT
# ============================================================

patient_ids = (
    notes_dedup["person_id"]
    .drop_duplicates()
    .tolist()
)

next_patient = next(
    person_id
    for person_id in patient_ids
    if person_id not in hierarchical_results
)

print("Next unfinished patient:", next_patient)

print(
    "Notes:",
    len(
        notes_dedup[
            notes_dedup["person_id"]
            == next_patient
        ]
    )
)

Next unfinished patient: 5e434d78-b2f6-4d88-b327-fff6ee50b901
Notes: 27


In [ ]:
# ============================================================
# COMPLETE PATIENT 20 ONLY
# ============================================================

person_id = next_patient

patient_df = notes_dedup[
    notes_dedup["person_id"] == person_id
].copy()

result = run_hierarchical_patient(
    person_id,
    patient_df,
)

hierarchical_results[
    person_id
] = result

with FINAL_RESULTS_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        hierarchical_results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved patient:", person_id)
print(
    "Total completed patients:",
    len(hierarchical_results)
)

Saved patient: 5e434d78-b2f6-4d88-b327-fff6ee50b901
Total completed patients: 20


In [ ]:
# ============================================================
# FREEZE FINAL 20-PATIENT STUDY COHORT
#
# These exact patients will be used across all four workflows.
# ============================================================

STUDY_PATIENT_IDS = list(
    hierarchical_results.keys()
)

assert len(STUDY_PATIENT_IDS) == 20, (
    f"Expected 20 completed patients, "
    f"found {len(STUDY_PATIENT_IDS)}"
)

COHORT_PATH = (
    PROJECT_ROOT
    / "data"
    / "patients"
    / "study_patient_ids.json"
)

COHORT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

with COHORT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        STUDY_PATIENT_IDS,
        file,
        indent=2,
    )

print("Frozen study cohort:", len(STUDY_PATIENT_IDS))
print("Saved:", COHORT_PATH)

for i, person_id in enumerate(
    STUDY_PATIENT_IDS,
    start=1,
):
    note_count = (
        notes_dedup["person_id"]
        == person_id
    ).sum()

    print(
        f"{i:02d} | {person_id} | "
        f"{note_count} notes"
    )

Frozen study cohort: 20
Saved: /Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/patients/study_patient_ids.json
01 | 028998ee-babc-4096-9b28-001bc2f9a84e | 15 notes
02 | 04df53ea-55c1-48d9-84a1-1f15c133b29b | 33 notes
03 | 05192757-942f-460d-b4ff-004ec39cc5ee | 23 notes
04 | 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf | 19 notes
05 | 0f438665-d430-4adb-8acc-c3beed9e4942 | 15 notes
06 | 136c7916-4f9b-4e5c-bf01-77e9d2c681a2 | 37 notes
07 | 137b8481-4f1d-4b7f-babd-20f7117023ad | 36 notes
08 | 1705dd0f-011a-492c-b006-b27e03f2f4ed | 14 notes
09 | 1dbe23dc-0d1e-431b-81eb-497282b46a14 | 17 notes
10 | 28570119-9cdc-4120-98c0-4edb76cf36a3 | 19 notes
11 | 29ea304f-821d-474e-81a1-394ca3945e02 | 17 notes
12 | 31f9612b-5a6b-48ea-887b-895772a83b99 | 22 notes
13 | 359014a1-10e6-4bd8-9ba7-513d021c971e | 34 notes
14 | 37b5ce4d-dcfd-4bb7-bee4-d597eb114703 | 26 notes
15 | 420df33b-0072-4124-b1c6-0589daef3677 | 24 notes
16 | 42149ae1-6a3c-471e-a002-cb7263e8bb8c | 25 notes
17 | 51f15281-8840-